# Study 868 — Global Curve-Slope Carry — the teardown

The Newey-West book *t*, the buy-and-hold benchmark, the 3,000-draw column-permutation placebo, the three-era robustness cut, the formation-window sweep, the costed backtest, and the 20-seed synthetic control.

In [1]:
R = {'start': '2007-01-31', 'end': '2026-06-30', 'n_etfs': 6, 'n_months': 215, 'fingerprint': '63122f29382a', 'bh_bps': 21.36, 'bh_ann': 2.56, 'bh_sharpe': 0.398, 'bh_t': 1.79, 'ytd_bps': -20.17, 'ytd_ann': -2.42, 'ytd_vol': 8.1, 'ytd_sharpe': -0.299, 'ytd_t_nw': -1.45, 'ytd_t1s': -1.27, 'ytd_hit': 0.447, 'ytd_long': 9.99, 'ytd_short': 30.16, 'ytd_pl_obs': -20.17, 'ytd_pl_mean': -0.23, 'ytd_pl_sd': 10.91, 'ytd_pl_p': 0.9387, 'ytd_e1_bps': -11.37, 'ytd_e1_t': -0.38, 'ytd_e2_bps': -40.24, 'ytd_e2_t': -3.02, 'ytd_e3_bps': 2.27, 'ytd_e3_t': 0.13, 'ytd_c5_net': -28.24, 'ytd_c5_cost': 8.07, 'ytd_c5_netS': -0.419, 'ytd_c5_t': -1.77, 'ytd_c10_net': -30.06, 'ytd_c10_cost': 9.89, 'ytd_c10_netS': -0.446, 'ytd_c10_t': -1.89, 'ytd_turn': 0.36, 'raw_bps': 3.16, 'raw_ann': 0.38, 'raw_vol': 8.79, 'raw_sharpe': 0.043, 'raw_t_nw': 0.2, 'raw_t1s': 0.18, 'raw_hit': 0.447, 'raw_long': 16.98, 'raw_short': 13.82, 'raw_pl_obs': 3.16, 'raw_pl_mean': -0.03, 'raw_pl_sd': 7.7, 'raw_pl_p': 0.439, 'raw_e1_bps': 41.59, 'raw_e1_t': 1.23, 'raw_e2_bps': -30.08, 'raw_e2_t': -1.7, 'raw_e3_bps': 7.17, 'raw_e3_t': 0.42, 'raw_c5_net': -4.81, 'raw_c5_netS': -0.066, 'raw_c5_t': -0.28, 'raw_c10_net': -6.53, 'raw_c10_cost': 9.69, 'raw_c10_netS': -0.089, 'raw_c10_t': -0.38, 'raw_turn': 0.34, 'w24_t': -1.7, 'w36_t': -1.45, 'w48_t': -0.46, 'w60_t': 0.29, 'null_mean_t': -0.05, 'null_fire': 2, 'planted_mean_t': 18.17, 'planted_sharpe': 3.5, 'planted_fire': 20}

## The headline — yield-to-duration vs raw carry vs just holding the bonds

A carry edge has to beat *owning the bonds*. Neither variant does — and the yield-to-duration sort is actively negative.

In [2]:
print(f"buy-and-hold      : {R['bh_bps']:+.2f} bps/mo  Sharpe {R['bh_sharpe']:.3f}  NW t = {R['bh_t']:+.2f}")
print(f"yield-to-duration : {R['ytd_bps']:+.2f} bps/mo  Sharpe {R['ytd_sharpe']:+.3f}  NW t = {R['ytd_t_nw']:+.2f}  (long {R['ytd_long']:+.2f} / short {R['ytd_short']:+.2f} bps)")
print(f"raw realized-yield: {R['raw_bps']:+.2f} bps/mo  Sharpe {R['raw_sharpe']:+.3f}  NW t = {R['raw_t_nw']:+.2f}  (long {R['raw_long']:+.2f} / short {R['raw_short']:+.2f} bps)")
print('  -> the yield-to-duration SHORT leg out-earns its LONG leg: the carry sort is inverted')

buy-and-hold      : +21.36 bps/mo  Sharpe 0.398  NW t = +1.79
yield-to-duration : -20.17 bps/mo  Sharpe -0.299  NW t = -1.45  (long +9.99 / short +30.16 bps)
raw realized-yield: +3.16 bps/mo  Sharpe +0.043  NW t = +0.20  (long +16.98 / short +13.82 bps)
  -> the yield-to-duration SHORT leg out-earns its LONG leg: the carry sort is inverted


## Placebo — permute which market feeds each rank (3,000 draws)

Break the carry->forward-return link; does the sort beat a random leg assignment?

In [3]:
print(f"yield-to-duration: observed {R['ytd_pl_obs']:+.2f} bps vs null {R['ytd_pl_mean']:+.2f} (sd {R['ytd_pl_sd']:.2f}) -> right-tail p = {R['ytd_pl_p']:.4f}")
print(f"raw realized-yield: observed {R['raw_pl_obs']:+.2f} bps vs null {R['raw_pl_mean']:+.2f} (sd {R['raw_pl_sd']:.2f}) -> right-tail p = {R['raw_pl_p']:.4f}")
print('  yield-to-duration observation sits DEEP in the wrong tail (p=0.94) -> beaten by random assignment')

yield-to-duration: observed -20.17 bps vs null -0.23 (sd 10.91) -> right-tail p = 0.9387
raw realized-yield: observed +3.16 bps vs null -0.03 (sd 7.70) -> right-tail p = 0.4390
  yield-to-duration observation sits DEEP in the wrong tail (p=0.94) -> beaten by random assignment


## Robustness — three eras and a formation-window sweep

In [4]:
print('era          yield-to-duration      raw realized-yield')
print(f"2010-2016    {R['ytd_e1_bps']:+7.2f} (t={R['ytd_e1_t']:+.2f})   {R['raw_e1_bps']:+7.2f} (t={R['raw_e1_t']:+.2f})")
print(f"2016-2021    {R['ytd_e2_bps']:+7.2f} (t={R['ytd_e2_t']:+.2f})   {R['raw_e2_bps']:+7.2f} (t={R['raw_e2_t']:+.2f})")
print(f"2021-2026    {R['ytd_e3_bps']:+7.2f} (t={R['ytd_e3_t']:+.2f})   {R['raw_e3_bps']:+7.2f} (t={R['raw_e3_t']:+.2f})")
print(f"window NW t (yield-to-duration): 24m {R['w24_t']:+.2f}  36m {R['w36_t']:+.2f}  48m {R['w48_t']:+.2f}  60m {R['w60_t']:+.2f}  (flips sign, never a robust positive)")

era          yield-to-duration      raw realized-yield
2010-2016     -11.37 (t=-0.38)    +41.59 (t=+1.23)
2016-2021     -40.24 (t=-3.02)    -30.08 (t=-1.70)
2021-2026      +2.27 (t=+0.13)     +7.17 (t=+0.42)
window NW t (yield-to-duration): 24m -1.70  36m -1.45  48m -0.46  60m +0.29  (flips sign, never a robust positive)


## The costed backtest — one-way turnover + short borrow

One-way cost x turnover per rebalance; short book pays 75 bps/yr borrow.

In [5]:
print(f"yield-to-duration  5 bps: gross {R['ytd_bps']:+.2f} -> net {R['ytd_c5_net']:+.2f} bps/mo (cost {R['ytd_c5_cost']:.2f}, net Sharpe {R['ytd_c5_netS']:+.3f}, t={R['ytd_c5_t']:+.2f})")
print(f"yield-to-duration 10 bps: gross {R['ytd_bps']:+.2f} -> net {R['ytd_c10_net']:+.2f} bps/mo (cost {R['ytd_c10_cost']:.2f}, net Sharpe {R['ytd_c10_netS']:+.3f}, t={R['ytd_c10_t']:+.2f})")
print(f"raw realized-yield  5 bps: gross {R['raw_bps']:+.2f} -> net {R['raw_c5_net']:+.2f} bps/mo (net Sharpe {R['raw_c5_netS']:+.3f}, t={R['raw_c5_t']:+.2f})")
print(f"raw realized-yield 10 bps: gross {R['raw_bps']:+.2f} -> net {R['raw_c10_net']:+.2f} bps/mo (cost {R['raw_c10_cost']:.2f}, net Sharpe {R['raw_c10_netS']:+.3f}, t={R['raw_c10_t']:+.2f})")

yield-to-duration  5 bps: gross -20.17 -> net -28.24 bps/mo (cost 8.07, net Sharpe -0.419, t=-1.77)
yield-to-duration 10 bps: gross -20.17 -> net -30.06 bps/mo (cost 9.89, net Sharpe -0.446, t=-1.89)
raw realized-yield  5 bps: gross +3.16 -> net -4.81 bps/mo (net Sharpe -0.066, t=-0.28)
raw realized-yield 10 bps: gross +3.16 -> net -6.53 bps/mo (cost 9.69, net Sharpe -0.089, t=-0.38)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire when every market yields the same, and must recover a planted carry spread.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from curve_slope_carry import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=868+s))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.010, seed=868))
print(f"planted (edge=0.010): NW t = {planted['t_nw']:+.2f}, Sharpe = {planted['sharpe']:.2f}")

null (edge=0), 8 seeds: NW t mean +0.50 (sd 0.94), |t|>=2 in 1/8
planted (edge=0.010): NW t = +18.09, Sharpe = 3.30


## Verdict

- **Signal — None.** The claimed global curve-slope carry premium does **not** appear. The yield-to-duration sort is **wrong-signed** (**-20.17 bps/mo**, NW *t* = **-1.45**, placebo *p* = 0.94 — beaten by random assignment), the raw-carry sort is flat (**+3.16 bps/mo**, NW *t* = +0.20, placebo *p* = 0.44), and both flip sign across eras and windows. The 20-seed synthetic control fires on a planted carry (mean *t* = +18.17, 2/20 on the null) — the null is real.
- **Tradability — Mirage.** Every variant loses money once costed (yield-to-duration net **-28.24 bps/mo** at 5 bps, raw carry net **-4.81 bps/mo** at 5 bps; net Sharpe < 0 throughout) — no costed net edge survives.